In [3]:
import pandas as pd

# ── Load data ────────────────────────────────────────────────────
mapping  = pd.read_excel(r"D:\Downloads\Mapping File APP Analyst Interview.xlsx")
sector   = pd.read_excel(r"D:\Downloads\Sector Data APP Analyst Interview.xlsx")
provider = pd.read_excel(r"D:\Downloads\Provider Data APP Analyst Interview.xlsx")

# ── Standardise year format ──────────────────────────────────────
sector['Academic Year'] = sector['Academic Year'].str.replace('/', '-')

# ── Add sector as a provider row ─────────────────────────────────
sector['UKPRN'] = 'SECTOR'
sector['Provider Name'] = 'Sector Average'
all_data = pd.concat([provider, sector], ignore_index=True)

# ── Clean attainment column ──────────────────────────────────────
pct_col = 'Percentage of students achieving first or upper-second class degrees'
all_data[pct_col] = pd.to_numeric(all_data[pct_col], errors='coerce')
# errors='coerce' turns [Data suppressed] into NaN automatically

# ── Calculate gaps ───────────────────────────────────────────────
results = []

group_cols = ['UKPRN', 'Provider Name', 'Academic Year', 'Student Characteristic Code']

for (ukprn, provider_name, year, char_code), group in all_data.groupby(group_cols):

    # Get the comparison group code (same for all rows in this group)
    comp_code = group['Comparison Group Student Attribute Code'].iloc[0]

    # Find the comparison group rate
    comp_row = group[group['Student Attribute Code'] == comp_code]
    if comp_row.empty or pd.isna(comp_row[pct_col].iloc[0]):
        continue  # skip if comparison rate is missing
    comp_rate = comp_row[pct_col].iloc[0]

    # Calculate gap for every attribute in this group
    for _, row in group.iterrows():
        target_rate = row[pct_col]
        if pd.isna(target_rate):
            continue  # skip suppressed data

        gap = comp_rate - target_rate  # positive = target group disadvantaged

        results.append({
            'UKPRN'              : ukprn,
            'Provider Name'      : provider_name,
            'Academic Year'      : year,
            'Char Code'          : char_code,
            'Attr Code'          : row['Student Attribute Code'],
            'Comp Code'          : comp_code,
            'Target Rate (%)'    : round(target_rate, 1),
            'Comparison Rate (%)': round(comp_rate, 1),
            'Attainment Gap (pp)': round(gap, 1),
        })

gaps_df = pd.DataFrame(results)

# ── Add readable labels from mapping file ────────────────────────
gaps_df = gaps_df.merge(
    mapping.rename(columns={
        'Student Charcteristic'     : 'Characteristic',
        'Student Attribute'         : 'Attribute',
        'Student Characteristic Code': 'Char Code',
        'Student Attribute Code'    : 'Attr Code'
    }),
    on=['Char Code', 'Attr Code'],
    how='left'
)

# ── Save full results ────────────────────────────────────────────
gaps_df.to_csv('all_attainment_gaps.csv', index=False)
print(f"Done — {len(gaps_df)} gap calculations saved to all_attainment_gaps.csv")

# ── Preview: UoN key gaps in 2023-24 ────────────────────────────
uon_latest = gaps_df[
    (gaps_df['UKPRN'] == 10007154) &
    (gaps_df['Academic Year'] == '2023-24') &
    (gaps_df['Attr Code'] != gaps_df['Comp Code'])  # exclude baseline vs itself
][['Characteristic', 'Attribute', 'Target Rate (%)', 'Comparison Rate (%)', 'Attainment Gap (pp)']]\
  .sort_values('Attainment Gap (pp)', ascending=False)

print("\nUoN Attainment Gaps — 2023-24:")
print(uon_latest.to_string(index=False))

Done — 2499 gap calculations saved to all_attainment_gaps.csv

UoN Attainment Gaps — 2023-24:
           Characteristic                    Attribute  Target Rate (%)  Comparison Rate (%)  Attainment Gap (pp)
                Ethnicity                        Black             55.5                 84.1                 28.6
English IMD Quintile 2019                        IMDQ1             59.2                 84.2                 25.0
          FSM Eligibility             Eligible for FSM             65.7                 79.7                 14.0
English IMD Quintile 2019                        IMDQ2             71.4                 84.2                 12.8
                Ethnicity                        Asian             72.4                 84.1                 11.7
                Ethnicity                        Other             73.8                 84.1                 10.3
English IMD Quintile 2019                        IMDQ3             77.4                 84.2                